# Script

In [1]:
%%writefile normalize_utils.py

import re


def normalize_half_brackets(text: str) -> str:
    """Remove half-bracket editorial marks."""
    for ch in "˹˺⸢⸣⌈⌉⌊⌋":
        text = text.replace(ch, "")
    return textcccccbngjchbtgdrfkjfrrblrrttglibcrcrunftndie
    


def normalize_heth(text: str) -> str:
    """ḫ → h, Ḫ → H."""
    return text.translate(str.maketrans("ḫḪ", "hH"))


def normalize_fractions(text: str) -> str:
    """Decimal floats, slash fractions, and superscript/subscript fractions → Unicode."""
    # Superscript/subscript fractions first (before subscript digits get converted)
    for src, dst in [
        ("¹/₂", "½"),
        ("¹/₃", "⅓"),
        ("²/₃", "⅔"),
        ("¹/₄", "¼"),
        ("³/₄", "¾"),
        ("¹/₆", "⅙"),
        ("⁵/₆", "⅚"),
        ("⁵/₈", "⅝"),
    ]:
        text = text.replace(src, dst)

    # Decimal fractions (longest prefix first to avoid partial matches)
    for prefix, uni in [
        ("0.8333", "⅚"),
        ("0.1666", "⅙"),
        ("0.6666", "⅔"),
        ("0.3333", "⅓"),
        ("0.75", "¾"),
        ("0.625", "⅝"),
        ("0.25", "¼"),
        ("0.5", "½"),
    ]:
        esc = re.escape(prefix)
        text = re.sub(rf"\b{esc}\d*\b", uni, text)
        esc_no_zero = re.escape(prefix[1:])
        text = re.sub(rf"(\d+){esc_no_zero}\d*\b", rf"\1 {uni}", text)

    # Slash fractions: 1/3, 1 / 3
    _frac = {
        (1, 2): "½",
        (1, 3): "⅓",
        (2, 3): "⅔",
        (1, 4): "¼",
        (3, 4): "¾",
        (1, 6): "⅙",
        (5, 6): "⅚",
        (5, 8): "⅝",
    }

    def _repl(m):
        return _frac.get((int(m.group(1)), int(m.group(2))), m.group(0))

    text = re.sub(r"(\d+)\s*/\s*(\d+)", _repl, text)

    # Spacing: 1½ → 1 ½
    text = re.sub(r"(\d)([½⅓⅔¼¾⅙⅚⅝])", r"\1 \2", text)
    return text


def normalize_subscripts(text: str) -> str:
    """Vowel subscripts → diacritics, then remaining subscripts → digits."""
    for src, dst in [
        ("a₂", "á"),
        ("e₂", "é"),
        ("i₂", "í"),
        ("u₂", "ú"),
        ("A₂", "Á"),
        ("E₂", "É"),
        ("I₂", "Í"),
        ("U₂", "Ú"),
    ]:
        text = text.replace(src, dst)
    for src, dst in [
        ("a₃", "à"),
        ("e₃", "è"),
        ("i₃", "ì"),
        ("u₃", "ù"),
        ("A₃", "À"),
        ("E₃", "È"),
        ("I₃", "Ì"),
        ("U₃", "Ù"),
    ]:
        text = text.replace(src, dst)
    text = text.translate(str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789"))
    return text


def normalize_macrons_tl(text: str) -> str:
    """Remove macron vowels from transliteration (not in test TL charset)."""
    return text.translate(str.maketrans("āēīūĀĒĪŪ", "aeiuAEIU"))


def normalize_brackets_tl(text: str) -> str:
    """Remove editorial brackets from transliteration, preserving content."""
    # <<text>> erasure removal (content removed entirely)
    text = re.sub(r"<<[^>]*>>", "", text)
    text = re.sub(r"«[^»]*»", "", text)
    text = re.sub(r"≪[^≫]*≫", "", text)
    # [text] square brackets — remove brackets, keep content
    text = re.sub(r"\[([^\]]*)\]", r"\1", text)
    # Unmatched brackets
    text = text.replace("[", "").replace("]", "")

    # <text> angle brackets — keep content, preserve <gap>/<big_gap>
    def _strip_angle(m):
        content = m.group(1)
        if content in ("gap", "big_gap"):
            return m.group(0)
        return content

    text = re.sub(r"<([^>]+)>", _strip_angle, text)
    return text


def normalize_determinatives(text: str) -> str:
    """Normalize determinative format: (d)→{d}, handle TÚG."""
    # Superscript determinatives
    for src, dst in [("ᵈ", "{d}"), ("ᵐ", "{m}"), ("ᶠ", "{f}"), ("ᵏⁱ", "{ki}")]:
        text = text.replace(src, dst)
    # Caret notation → curly
    for src, dst in [("^d", "{d}"), ("^ki", "{ki}"), ("^m", "{m}"), ("^f", "{f}")]:
        text = text.replace(src, dst)
    # Parenthetical → curly for core determinatives
    for src, dst in [("(d)", "{d}"), ("(ki)", "{ki}"), ("(m)", "{m}"), ("(f)", "{f}")]:
        text = text.replace(src, dst)
    # Variant curly forms → standard
    for src, dst in [("{D}", "{d}"), ("{DINGIR}", "{d}"), ("{KI}", "{ki}"), ("{M}", "{m}"), ("{1}", "{m}"), ("{MÍ}", "{f}"), ("{MI}", "{f}"), ("{MUNUS}", "{f}")]:
        text = text.replace(src, dst)
    # TÚG: host says no brackets in test data
    for src in ["{TÚG}", "{túg}", "{tug}", "(TÚG)", "(túg)", "(tug)"]:
        text = text.replace(src, "TÚG")
    return text


def normalize_silver(text: str) -> str:
    """KÙ.B → KÙ.BABBAR, KB → KÙ.BABBAR."""
    text = re.sub(r"KÙ\.B\.", "KÙ.BABBAR", text)
    text = re.sub(r"KÙ\.B\b", "KÙ.BABBAR", text)
    text = re.sub(r"\bKB\b", "KÙ.BABBAR", text)
    return text


def normalize_editorial_marks(text: str) -> str:
    """Strip editorial marks (!, ?, *) from transliteration."""
    for ch in "!?*":
        text = text.replace(ch, "")
    return text


def normalize_dividers(text: str) -> str:
    """Remove colon/slash word dividers and section marks."""
    text = text.replace(" : ", " ")
    text = re.sub(r"\s+:$", "", text)
    text = re.sub(r"^:\s+", "", text)
    text = text.replace("§", "")
    text = text.replace("=", "-")
    text = text.replace("--", "-")
    text = re.sub(r"-/\s*", "-", text)
    text = re.sub(r"\s+/\s+", " ", text)
    return text


def normalize_turkish_chars(text: str) -> str:
    """Map Turkish chars not in test charset to Akkadological equivalents."""
    text = text.replace("Ş", "Š")
    text = text.replace("ș", "ṣ")
    return text


def normalize_dashes(text: str) -> str:
    """Normalize dash variants to hyphen."""
    for ch in "\u2014\u2013\u2012":  # em-dash, en-dash, figure dash
        text = text.replace(ch, "-")
    return text


def normalize_glottal_stops(text: str) -> str:
    """Strip glottal stop and apostrophe variants from transliteration."""
    for ch in "ˀʾˈʼ᾿ˁˊˋˇꜥꜣʿˮʹꞌ`´":
        text = text.replace(ch, "")
    return text


def normalize_gaps(text: str) -> str:
    """Convert damage markers to <gap> and merge adjacent gaps."""
    text = text.replace("…", "<gap>")
    text = re.sub(r"\.{3,}", "<gap>", text)
    text = re.sub(r"\[x(\s+x)*\]", "<gap>", text)
    text = re.sub(r"\((?:large\s+)?break\)", "<gap>", text)
    text = re.sub(r"\(\d+\s+broken\s+lines?\)", "<gap>", text)
    text = re.sub(r"(\.\s){2,}\.?", "<gap>", text)
    text = re.sub(r"\b(?:ca\.\s*)?\d+\s+lines?\s+(?:broken|destroyed|lost|missing)(?:\s+away)?\b", "<gap>", text, flags=re.IGNORECASE)
    text = re.sub(r"\b(?:one|two|three|four|five|six|seven|eight|nine|ten)\s+lines?\s+(?:broken|destroyed|lost|missing)(?:\s+away)?\b", "<gap>", text, flags=re.IGNORECASE)
    text = re.sub(r"\bbreak\s+of\s+(?:\w+\s+)?lines?\b", "<gap>", text, flags=re.IGNORECASE)
    text = re.sub(
        r"(?<![a-zA-ZšṣṭáàéèíìúùÁÀÉÈÍÌÚÙ<-])x(?![a-zA-ZšṣṭáàéèíìúùÁÀÉÈÍÌÚÙ>-])",
        "<gap>",
        text,
    )
    text = text.replace("<big_gap>", "<gap>")
    text = text.replace("(<gap>)", "<gap>")
    # Merge adjacent
    prev = None
    while prev != text:
        prev = text
        text = re.sub(r"<gap>\s+<gap>", "<gap>", text)
        text = text.replace("<gap>-<gap>", "<gap>")
    return text


def normalize_whitespace(text: str) -> str:
    """Collapse multiple spaces, strip."""
    return re.sub(r"  +", " ", text).strip()


# ── Pipeline ─────────────────────────────────────────────────────────


def normalize_transliteration(text: str) -> str:
    """Normalize transliteration for pre-training."""
    try:
        text = normalize_half_brackets(text)
        text = normalize_brackets_tl(text)
        text = normalize_determinatives(text)
        text = normalize_silver(text)
        text = normalize_editorial_marks(text)
        text = normalize_dividers(text)
        text = normalize_turkish_chars(text)
        text = normalize_dashes(text)
        text = normalize_heth(text)
        text = normalize_fractions(text)
        text = normalize_subscripts(text)
        text = normalize_macrons_tl(text)
        text = normalize_glottal_stops(text)
        text = normalize_gaps(text)
        text = normalize_whitespace(text)
        return text
    except Exception as e:
        print(e)
        return text


# ── Translation-specific normalizers ─────────────────────────────────


def normalize_linebreaks_tr(text: str) -> str:
    """Rejoin OCR line-break hyphens in translation: mer- chant → merchant."""
    return re.sub(r"(?<=\s)([a-z]+)- ([a-z]+)", r"\1\2", text)


def normalize_brackets_tr(text: str) -> str:
    """Remove brackets from translation, keep content."""
    # <<text>> erasure removal
    text = re.sub(r"<<[^>]*>>", "", text)

    # <text> angle brackets — keep content, preserve <gap>/<big_gap>
    def _strip_angle(m):
        content = m.group(1)
        if content in ("gap", "big_gap"):
            return m.group(0)
        return content

    text = re.sub(r"<([^>]+)>", _strip_angle, text)
    # [text] square brackets — keep content
    text = re.sub(r"\[([^\]]*)\]", r"\1", text)
    text = text.replace("[", "").replace("]", "")
    return text


def normalize_quotes(text: str) -> str:
    """Curly/smart quotes → straight quotes."""
    for src, dst in [("\u201c", '"'), ("\u201d", '"'), ("\u201e", '"'), ("\u2018", "'"), ("\u2019", "'"), ("\u201a", "'")]:
        text = text.replace(src, dst)
    return text


def normalize_macrons_tr(text: str) -> str:
    """Capital macrons → plain. Keep lowercase ā ē ī ū (in test charset)."""
    return text.translate(str.maketrans("ĀĒĪŪ", "AEIU"))


def normalize_circumflex_tr(text: str) -> str:
    """Circumflex → macron for û ê î. Keep â (in test charset)."""
    for src, dst in [("û", "ū"), ("ê", "ē"), ("î", "ī"), ("Û", "Ū"), ("Ê", "Ē"), ("Î", "Ī")]:
        text = text.replace(src, dst)
    return text


def normalize_glottal_tr(text: str) -> str:
    """Glottal stop variants → straight apostrophe in translation."""
    for ch in "ˀʾˈʼ᾿ˁꜥꜣʿˮʹꞌ":
        text = text.replace(ch, "'")
    return text


def normalize_grammar(text: str) -> str:
    """Remove grammatical annotations from translation."""
    for pattern in [
        r"\(fem\.\s*plur\.\)",
        r"\(fem\.\s*pl\.\)",
        r"\(fem\.\s*sing\.\)",
        r"\(fem\.\)",
        r"\(plur\.\)",
        r"\(sing\.\)",
        r"\(masc\.\)",
        r"\(pl\.\)",
        r"\(m\.\)",
        r"\(f\.\)",
        r"\(\?\)",
        r"\bfem\.\s",
        r"\bsing\.\s",
        r"\bpl\.\s",
        r"\bplural\b",
    ]:
        text = re.sub(pattern, "", text)
    return text


def normalize_punctuation(text: str) -> str:
    """Remove space before punctuation."""
    return re.sub(r"\s+([.,;:?!])", r"\1", text)


def normalize_double_dots(text: str) -> str:
    """Remove stray double dots (not part of longer sequences)."""
    return re.sub(r"(?<!\.)\.\.(?!\.)", "", text)


def normalize_hyphen_terms(text: str) -> str:
    """Replace abbreviated hyphenated terms per host."""
    text = text.replace(" -gold", " pašallum gold")
    text = text.replace(" -tax", " šadduātum tax")
    text = text.replace(" -textiles", " kutānum textiles")
    return text


def normalize_pn(text: str) -> str:
    """Literal PN token → <gap>."""
    return re.sub(r"\bPN\b", "<gap>", text)


def normalize_months(text: str) -> str:
    """Month Roman numeral → integer."""
    _roman = {"XII": 12, "XI": 11, "X": 10, "IX": 9, "VIII": 8, "VII": 7, "VI": 6, "V": 5, "IV": 4, "III": 3, "II": 2, "I": 1}

    def _repl(m):
        prefix = m.group(1)
        roman = m.group(2)
        return f"{prefix} {_roman[roman]}" if roman in _roman else m.group(0)

    return re.sub(r"\b(month|Month)\s+(XII|XI|X|IX|VIII|VII|VI|V|IV|III|II|I)\b", _repl, text)


# ── Translation pipeline ─────────────────────────────────────────────


def normalize_translation(text: str) -> str:
    """Normalize translation for pre-training."""
    try:
        text = normalize_half_brackets(text)
        text = normalize_linebreaks_tr(text)
        text = normalize_brackets_tr(text)
        text = normalize_heth(text)
        text = normalize_quotes(text)
        text = normalize_macrons_tr(text)
        text = normalize_circumflex_tr(text)
        text = normalize_glottal_tr(text)
        text = normalize_grammar(text)
        text = normalize_hyphen_terms(text)
        text = normalize_pn(text)
        text = normalize_double_dots(text)
        text = normalize_months(text)
        text = normalize_fractions(text)
        text = normalize_subscripts(text)
        text = normalize_gaps(text)
        text = text.replace("--", "-")
        text = normalize_punctuation(text)
        text = normalize_whitespace(text)
        return text
    
    except Exception as e:
        print(e)
        return text


Writing normalize_utils.py


In [2]:
%%writefile run_inference.py

import argparse
import math
from dataclasses import dataclass
from pathlib import Path

import pandas as pd
import sacrebleu
import torch
from accelerate import Accelerator
from omegaconf import OmegaConf
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer, DataCollatorWithPadding, GenerationConfig, T5ForConditionalGeneration

import sys
sys.path.insert(0, "./")
from normalize_utils import normalize_translation, normalize_transliteration

# ── MBR Decoding ──────────────────────────────────────────────────────


def comp_metric_utility(hypothesis, reference):
    """Sentence-level sqrt(BLEU * chrF++) — same as competition metric."""
    bleu = sacrebleu.sentence_bleu(hypothesis, [reference]).score
    chrf = sacrebleu.sentence_chrf(hypothesis, [reference], word_order=2).score
    return math.sqrt(max(bleu, 0.0) * max(chrf, 0.0))


def mbr_select(candidates, utility_fn=None):
    """Return index of candidate with highest average pairwise utility."""
    if utility_fn is None:
        utility_fn = comp_metric_utility
    n = len(candidates)
    if n == 1:
        return 0
    scores = []
    for i in range(n):
        total = sum(utility_fn(candidates[i], candidates[j]) for j in range(n) if j != i)
        scores.append(total / (n - 1))
    return max(range(n), key=lambda i: scores[i])


@torch.no_grad()
def generate_mbr_predictions(model, tokenizer, eval_dataloader, accelerator, generation_config, mbr_n=16):
    model.eval()
    accelerator.print(f"MBR decoding: N={mbr_n} candidates per input")
    accelerator.wait_for_everyone()

    unwrapped_model = accelerator.unwrap_model(model)
    pad_id = tokenizer.pad_token_id or 0
    all_predictions = []

    progress_bar = tqdm(eval_dataloader, desc="MBR Decoding", disable=not accelerator.is_local_main_process)

    for batch in progress_bar:
        local_batch_size = batch["input_ids"].shape[0]

        generated_ids = unwrapped_model.generate(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            generation_config=generation_config,
        )

        all_candidates = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

        best_token_ids = []
        for i in range(local_batch_size):
            group = all_candidates[i * mbr_n : (i + 1) * mbr_n]
            best_idx = mbr_select(group)
            best_token_ids.append(generated_ids[i * mbr_n + best_idx])

        selected = torch.stack(best_token_ids, dim=0)
        selected = accelerator.pad_across_processes(selected, dim=1, pad_index=pad_id)
        selected = accelerator.gather_for_metrics(selected)

        predictions = tokenizer.batch_decode(selected, skip_special_tokens=True)
        all_predictions.extend(predictions)

    accelerator.wait_for_everyone()
    return all_predictions


# ── Dataset & Collator ───────────────────────────────────────────────


def get_tokenizer(cfg):
    tokenizer = AutoTokenizer.from_pretrained(cfg.model.backbone_path)
    return tokenizer


class DPCDataset(Dataset):
    """
    Dataset class for Deep Past Challenge - Translate Akkadian to English
    """

    def __init__(self, cfg, df, id_col="oare_id"):
        self.cfg = cfg
        self.df = df
        self.id_col = id_col

        self.tokenizer = get_tokenizer(cfg)

    def __len__(self):
        return len(self.df)

    def tokenize(self, texts):
        return self.tokenizer(texts, padding=False, truncation=True, max_length=self.cfg.model.max_length, return_length=True, add_special_tokens=True)

    def pre_process(self, row):
        ak_en_template = "Translate Akkadian to English: {source}"

        return [ak_en_template.format(source=row["transliteration"])], [None]

    def __getitem__(self, idx):
        data = self.df.iloc[idx].to_dict()
        this_id = data[self.id_col]
        sources, targets = self.pre_process(data)
        tx_src = self.tokenize(sources)

        return dict(id=this_id, input_ids=tx_src["input_ids"], attention_mask=tx_src["attention_mask"])


@dataclass
class DPCCollator(DataCollatorWithPadding):
    """
    Data collector for DPC dataset
    """

    tokenizer = None
    padding = True
    max_length = None
    pad_to_multiple_of = None

    def __call__(self, features):
        encoder_features = {"input_ids": [], "attention_mask": []}
        labels = []

        for feature in features:
            encoder_features["input_ids"].extend(feature["input_ids"])
            encoder_features["attention_mask"].extend(feature["attention_mask"])

            if "labels" in feature:
                labels.extend(feature["labels"])

        if len(labels) > 0:
            assert len(labels) == len(encoder_features["input_ids"])

        batch = self.tokenizer.pad(
            encoder_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        # labels
        if len(labels) > 0:
            labels_batch = self.tokenizer.pad({"input_ids": labels}, padding=self.padding, max_length=self.max_length, pad_to_multiple_of=self.pad_to_multiple_of, return_tensors=None)
            labels = torch.tensor(labels_batch["input_ids"], dtype=torch.int64)
            # Mask padding with -100 (ignored in loss)
            labels[labels == self.tokenizer.pad_token_id] = -100
            batch["labels"] = labels
        return batch


def show_batch(batch, tokenizer, print_fn=print, **kwargs):
    bs = batch["input_ids"].size(0)
    print_fn(f"batch size: {bs}")

    print_fn(f"shape of input_ids: {batch['input_ids'].shape}")
    print_fn(f"shape of attention_mask: {batch['attention_mask'].shape}")
    if "labels" in batch.keys():
        print_fn(f"shape of labels: {batch['labels'].shape}")

    print_fn("\n\n")
    max_n = kwargs.get("n", 5)

    for idx in range(bs):
        print_fn(f"\n\n===== Example {idx + 1} =====\n\n")
        print_fn(f"## Input:\n\n{tokenizer.decode(batch['input_ids'][idx], skip_special_tokens=False)}\n\n")

        if "labels" in batch.keys():
            labels = batch["labels"][idx]
            labels[labels == -100] = tokenizer.pad_token_id
            print_fn(f"## Output:\n\n{tokenizer.decode(labels, skip_special_tokens=False)}")

        if idx >= max_n:
            break


# ── Generation ────────────────────────────────────────────────────────


@torch.no_grad()
def generate_predictions(model, tokenizer, eval_dataloader, accelerator, generation_config):
    model.eval()
    accelerator.print("Generating predictions...")
    accelerator.wait_for_everyone()

    unwrapped_model = accelerator.unwrap_model(model)

    all_predictions = []

    progress_bar = tqdm(eval_dataloader, desc="Evaluating", disable=not accelerator.is_local_main_process)

    for batch in progress_bar:
        # Generate predictions
        generated_ids = unwrapped_model.generate(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], generation_config=generation_config)

        # Pad and gather across processes
        generated_ids = accelerator.pad_across_processes(generated_ids, dim=1, pad_index=tokenizer.pad_token_id or 0)
        generated_ids = accelerator.gather_for_metrics(generated_ids)

        # Decode on main process
        predictions = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        all_predictions.extend(predictions)

    accelerator.wait_for_everyone()

    return all_predictions


def get_generation_config(cfg, tokenizer):
    return GenerationConfig(
        max_new_tokens=cfg.generation.max_new_tokens,
        do_sample=cfg.generation.do_sample,
        top_k=cfg.generation.top_k,
        top_p=cfg.generation.top_p,
        temperature=cfg.generation.temperature,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )


# ── Main ──────────────────────────────────────────────────────────────


def main(cfg, save_path):
    accelerator = Accelerator()

    test_df = pd.read_csv("/kaggle/input/deep-past-initiative-machine-translation/test.csv")
    test_df["transliteration"] = test_df["transliteration"].apply(normalize_transliteration)

    infer_ds = DPCDataset(cfg, test_df, id_col="id")
    tokenizer = infer_ds.tokenizer
    data_collator = DPCCollator(tokenizer=tokenizer, pad_to_multiple_of=16)

    infer_dl = DataLoader(
        infer_ds,
        batch_size=cfg.predict_params.per_device_eval_batch_size,
        shuffle=False,
        collate_fn=data_collator,
    )

    accelerator.print("showing batch...")
    for idx, b in enumerate(infer_dl):
        show_batch(b, tokenizer, task="infer", print_fn=accelerator.print, n=2)
        break

    model = T5ForConditionalGeneration.from_pretrained(cfg.model.backbone_path)
    model, infer_dl = accelerator.prepare(model, infer_dl)

    use_mbr = cfg.generation.get("mbr", False)
    mbr_n = cfg.generation.get("mbr_n", 16)

    if use_mbr:
        mbr_num_beams = cfg.generation.get("mbr_num_beams", 0)
        if mbr_num_beams > 0:
            generation_config = GenerationConfig(
                max_new_tokens=cfg.generation.max_new_tokens,
                use_cache=True,
                do_sample=False,
                num_beams=mbr_num_beams,
                num_return_sequences=mbr_n,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        else:
            generation_config = GenerationConfig(
                max_new_tokens=cfg.generation.max_new_tokens,
                use_cache=True,
                do_sample=True,
                temperature=cfg.generation.get("mbr_temperature", 0.5),
                top_p=cfg.generation.get("mbr_top_p", 0.9),
                top_k=cfg.generation.get("mbr_top_k", 0),
                num_return_sequences=mbr_n,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        predictions = generate_mbr_predictions(model=model, tokenizer=tokenizer, eval_dataloader=infer_dl, accelerator=accelerator, generation_config=generation_config, mbr_n=mbr_n)
    else:
        generation_config = get_generation_config(cfg, tokenizer)
        predictions = generate_predictions(model=model, tokenizer=tokenizer, eval_dataloader=infer_dl, accelerator=accelerator, generation_config=generation_config)

    predictions_df = pd.DataFrame({"id": test_df["id"], "translation": predictions, "transliteration": test_df["transliteration"]})
    predictions_df["translation"] = predictions_df["translation"].apply(normalize_translation)
    
    predictions_df.to_parquet(save_path)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--config_path", type=str, required=True)
    ap.add_argument("--save_path", type=str, required=True)
    args = ap.parse_args()

    cfg = OmegaConf.load(args.config_path)

    Path(args.save_path).parent.mkdir(parents=True, exist_ok=True)

    # run
    main(cfg, args.save_path)

Writing run_inference.py


In [3]:
%%writefile run_inference_xl.py

import argparse
import math
from dataclasses import dataclass
from pathlib import Path

import pandas as pd
import sacrebleu
import torch
from omegaconf import OmegaConf
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer, DataCollatorWithPadding, GenerationConfig, T5ForConditionalGeneration

import sys
sys.path.insert(0, "./")
from normalize_utils import normalize_translation, normalize_transliteration

# ── MBR Decoding ──────────────────────────────────────────────────────


def comp_metric_utility(hypothesis, reference):
    """Sentence-level sqrt(BLEU * chrF++) — same as competition metric."""
    bleu = sacrebleu.sentence_bleu(hypothesis, [reference]).score
    chrf = sacrebleu.sentence_chrf(hypothesis, [reference], word_order=2).score
    return math.sqrt(max(bleu, 0.0) * max(chrf, 0.0))


def mbr_select(candidates, utility_fn=None):
    """Return index of candidate with highest average pairwise utility."""
    if utility_fn is None:
        utility_fn = comp_metric_utility
    n = len(candidates)
    if n == 1:
        return 0
    scores = []
    for i in range(n):
        total = sum(utility_fn(candidates[i], candidates[j]) for j in range(n) if j != i)
        scores.append(total / (n - 1))
    return max(range(n), key=lambda i: scores[i])


@torch.no_grad()
def generate_mbr_predictions(model, tokenizer, eval_dataloader, generation_config, mbr_n=16):
    model.eval()
    print(f"MBR decoding: N={mbr_n} candidates per input")

    all_predictions = []
    progress_bar = tqdm(eval_dataloader, desc="MBR Decoding")

    for batch in progress_bar:
        input_ids = batch["input_ids"].to(model.device)
        attention_mask = batch["attention_mask"].to(model.device)
        local_batch_size = input_ids.shape[0]

        generated_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            generation_config=generation_config,
        )

        all_candidates = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

        for i in range(local_batch_size):
            group = all_candidates[i * mbr_n : (i + 1) * mbr_n]
            best_idx = mbr_select(group)
            all_predictions.append(group[best_idx])

    return all_predictions


# ── Dataset & Collator ───────────────────────────────────────────────


def get_tokenizer(cfg):
    tokenizer = AutoTokenizer.from_pretrained(cfg.model.backbone_path)
    return tokenizer


class DPCDataset(Dataset):
    """
    Dataset class for Deep Past Challenge - Translate Akkadian to English
    """

    def __init__(self, cfg, df, id_col="oare_id"):
        self.cfg = cfg
        self.df = df
        self.id_col = id_col

        self.tokenizer = get_tokenizer(cfg)

    def __len__(self):
        return len(self.df)

    def tokenize(self, texts):
        return self.tokenizer(texts, padding=False, truncation=True, max_length=self.cfg.model.max_length, return_length=True, add_special_tokens=True)

    def pre_process(self, row):
        ak_en_template = "Translate Akkadian to English: {source}"

        return [ak_en_template.format(source=row["transliteration"])], [None]

    def __getitem__(self, idx):
        data = self.df.iloc[idx].to_dict()
        this_id = data[self.id_col]
        sources, targets = self.pre_process(data)
        tx_src = self.tokenize(sources)

        return dict(id=this_id, input_ids=tx_src["input_ids"], attention_mask=tx_src["attention_mask"])


@dataclass
class DPCCollator(DataCollatorWithPadding):
    """
    Data collector for DPC dataset
    """

    tokenizer = None
    padding = True
    max_length = None
    pad_to_multiple_of = None

    def __call__(self, features):
        encoder_features = {"input_ids": [], "attention_mask": []}
        labels = []

        for feature in features:
            encoder_features["input_ids"].extend(feature["input_ids"])
            encoder_features["attention_mask"].extend(feature["attention_mask"])

            if "labels" in feature:
                labels.extend(feature["labels"])

        if len(labels) > 0:
            assert len(labels) == len(encoder_features["input_ids"])

        batch = self.tokenizer.pad(
            encoder_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        # labels
        if len(labels) > 0:
            labels_batch = self.tokenizer.pad({"input_ids": labels}, padding=self.padding, max_length=self.max_length, pad_to_multiple_of=self.pad_to_multiple_of, return_tensors=None)
            labels = torch.tensor(labels_batch["input_ids"], dtype=torch.int64)
            # Mask padding with -100 (ignored in loss)
            labels[labels == self.tokenizer.pad_token_id] = -100
            batch["labels"] = labels
        return batch


def show_batch(batch, tokenizer, print_fn=print, **kwargs):
    bs = batch["input_ids"].size(0)
    print_fn(f"batch size: {bs}")

    print_fn(f"shape of input_ids: {batch['input_ids'].shape}")
    print_fn(f"shape of attention_mask: {batch['attention_mask'].shape}")
    if "labels" in batch.keys():
        print_fn(f"shape of labels: {batch['labels'].shape}")

    print_fn("\n\n")
    max_n = kwargs.get("n", 5)

    for idx in range(bs):
        print_fn(f"\n\n===== Example {idx + 1} =====\n\n")
        print_fn(f"## Input:\n\n{tokenizer.decode(batch['input_ids'][idx], skip_special_tokens=False)}\n\n")

        if "labels" in batch.keys():
            labels = batch["labels"][idx]
            labels[labels == -100] = tokenizer.pad_token_id
            print_fn(f"## Output:\n\n{tokenizer.decode(labels, skip_special_tokens=False)}")

        if idx >= max_n:
            break


# ── Generation ────────────────────────────────────────────────────────


@torch.no_grad()
def generate_predictions(model, tokenizer, eval_dataloader, generation_config):
    model.eval()
    print("Generating predictions...")

    all_predictions = []
    progress_bar = tqdm(eval_dataloader, desc="Evaluating")

    for batch in progress_bar:
        input_ids = batch["input_ids"].to(model.device)
        attention_mask = batch["attention_mask"].to(model.device)

        generated_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            generation_config=generation_config,
        )

        predictions = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        all_predictions.extend(predictions)

    return all_predictions


def get_generation_config(cfg, tokenizer):
    return GenerationConfig(
        max_new_tokens=cfg.generation.max_new_tokens,
        do_sample=cfg.generation.do_sample,
        top_k=cfg.generation.top_k,
        top_p=cfg.generation.top_p,
        temperature=cfg.generation.temperature,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )


# ── Main ──────────────────────────────────────────────────────────────


def main(cfg, save_path):
    test_df = pd.read_csv("/kaggle/input/deep-past-initiative-machine-translation/test.csv")
    # test_df = pd.read_csv("/kaggle/input/deep-past-initiative-machine-translation/train.csv").head(128).rename(columns={'oare_id': 'id'})
    # test_df['transliteration'] = test_df['transliteration'].apply(lambda x: x[:80])
    test_df["transliteration"] = test_df["transliteration"].apply(normalize_transliteration)

    infer_ds = DPCDataset(cfg, test_df, id_col="id")
    tokenizer = infer_ds.tokenizer
    data_collator = DPCCollator(tokenizer=tokenizer, pad_to_multiple_of=16)

    infer_dl = DataLoader(
        infer_ds,
        batch_size=cfg.predict_params.per_device_eval_batch_size,
        shuffle=False,
        collate_fn=data_collator,
    )

    print("showing batch...")
    for idx, b in enumerate(infer_dl):
        show_batch(b, tokenizer, n=2)
        break

    model = T5ForConditionalGeneration.from_pretrained(
        cfg.model.backbone_path,
        dtype=torch.float16,
        device_map="auto",
    )

    use_mbr = cfg.generation.get("mbr", False)
    mbr_n = cfg.generation.get("mbr_n", 16)

    if use_mbr:
        mbr_num_beams = cfg.generation.get("mbr_num_beams", 0)
        if mbr_num_beams > 0:
            generation_config = GenerationConfig(
                max_new_tokens=cfg.generation.max_new_tokens,
                use_cache=True,
                do_sample=False,
                num_beams=mbr_num_beams,
                num_return_sequences=mbr_n,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        else:
            generation_config = GenerationConfig(
                max_new_tokens=cfg.generation.max_new_tokens,
                use_cache=True,
                do_sample=True,
                temperature=cfg.generation.get("mbr_temperature", 0.5),
                top_p=cfg.generation.get("mbr_top_p", 0.9),
                top_k=cfg.generation.get("mbr_top_k", 0),
                num_return_sequences=mbr_n,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        predictions = generate_mbr_predictions(model=model, tokenizer=tokenizer, eval_dataloader=infer_dl, generation_config=generation_config, mbr_n=mbr_n)
    else:
        generation_config = get_generation_config(cfg, tokenizer)
        predictions = generate_predictions(model=model, tokenizer=tokenizer, eval_dataloader=infer_dl, generation_config=generation_config)

    predictions_df = pd.DataFrame({"id": test_df["id"], "translation": predictions})
    predictions_df["translation"] = predictions_df["translation"].apply(normalize_translation)

    predictions_df.to_parquet(save_path)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--config_path", type=str, required=True)
    ap.add_argument("--save_path", type=str, required=True)
    args = ap.parse_args()

    cfg = OmegaConf.load(args.config_path)

    Path(args.save_path).parent.mkdir(parents=True, exist_ok=True)

    # run
    main(cfg, args.save_path)

Writing run_inference_xl.py


# Config

In [4]:
%%writefile conf_m1.yaml

model:
    backbone_path: "/kaggle/input/models/conjuring92/dpc_byt5_b2_ns/transformers/default/1"
    max_length: 1024
    num_proc: 2

predict_params:
    per_device_eval_batch_size: 1

generation:
    max_new_tokens: 512
    do_sample: true
    top_k: 50
    top_p: 1.0
    temperature: 0.0
    mbr: true
    mbr_n: 1
    mbr_num_beams: 8

Writing conf_m1.yaml


In [5]:
%%writefile conf_m2.yaml

model:
    backbone_path: "/kaggle/input/models/conjuring92/dpc_a05_14k_soup/transformers/default/1"
    max_length: 1024
    num_proc: 2

predict_params:
    per_device_eval_batch_size: 1

generation:
    max_new_tokens: 512
    do_sample: true
    top_k: 50
    top_p: 1.0
    temperature: 0.0
    mbr: true
    mbr_n: 1
    mbr_num_beams: 8

Writing conf_m2.yaml


In [6]:
%%writefile conf_m3.yaml

model:
    backbone_path: "/kaggle/input/models/conjuring92/dpc_byt5_xl/transformers/default/4"
    max_length: 1024
    num_proc: 2

predict_params:
    per_device_eval_batch_size: 1

generation:
    max_new_tokens: 512
    do_sample: true
    top_k: 50
    top_p: 1.0
    temperature: 0.0
    mbr: true
    mbr_n: 1
    mbr_num_beams: 8

Writing conf_m3.yaml


# Infer

In [7]:
%%time
!accelerate launch --multi_gpu --num_processes=2 run_inference.py --config_path "./conf_m1.yaml" --save_path "./model_outputs/pred_m1.parquet"

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [8]:
# %%time
# !accelerate launch --multi_gpu --num_processes=2 run_inference.py --config_path "./conf_m2.yaml" --save_path "./model_outputs/pred_m2.parquet"

In [9]:
%%time
!python run_inference_xl.py --config_path "./conf_m3.yaml" --save_path "./model_outputs/pred_m3.parquet"

2026-03-22 20:49:28.837466: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774212568.861552     141 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774212568.869393     141 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774212568.890151     141 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774212568.890180     141 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774212568.890184     141 computation_placer.cc:177] computation placer alr

# RM

In [10]:
%%writefile run_rm.py

import argparse
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import vllm
from omegaconf import OmegaConf

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_ATTENTION_BACKEND"] = "TRITON_ATTN"

# Token IDs for Qwen3-8B (from training)
TOK_A = 32
TOK_B = 33
TOK_EQUAL = 96024

SYS_PROMPT = """You are an expert judge comparing two Akkadian-to-English translations of the same source text. The transliteration is the ground truth — a syllable-by-syllable representation of an Old Assyrian cuneiform tablet (circa 1950-1750 BCE).

Your task: determine which translation is more faithful to the transliteration, or whether they are effectively equal.

Key evaluation criteria (in order of importance):
1. SEMANTIC FIDELITY: Who did what to whom, under what conditions, with what consequences. Reversing a debt relationship or dropping a negation is catastrophic.
2. NAME ACCURACY: Personal names must be derivable from the syllables in the transliteration. Minor spelling variants (macrons, gemination) are acceptable.
3. NUMBERS AND QUANTITIES: Weights, counts, and amounts must be preserved exactly. 1 mina = 60 shekels, 1 talent = 60 minas.
4. COMPLETENESS: A translation covering all content is better than one that stops midway. But fabrication is worse than omission.
5. ALIGNMENT: Every element in the transliteration should have a corresponding element in the translation.

Pick A if Translation A is more faithful. Pick B if Translation B is more faithful. Pick EQUAL if both are substantively equivalent."""

USER_TEMPLATE = """Transliteration: {transliteration}

Translation A: {translation_a}

Translation B: {translation_b}

Which translation is more faithful to the transliteration? Respond with only: A, B, or EQUAL."""


def get_probs(logprobs):
    """Extract softmax probabilities for A/B/EQUAL from vLLM logprobs."""
    tok_map = {TOK_A: 0, TOK_B: 1, TOK_EQUAL: 2}
    logits = np.full(3, -20.0)

    for token_id, logprob_obj in logprobs.items():
        if token_id in tok_map:
            logits[tok_map[token_id]] = logprob_obj.logprob

    exp_logits = np.exp(logits - np.max(logits))
    return exp_logits / np.sum(exp_logits)


def build_prompts(df, tokenizer):
    prompts = []
    for _, row in df.iterrows():
        user_message = USER_TEMPLATE.format(
            transliteration=row["transliteration"],
            translation_a=row["translation_a"],
            translation_b=row["translation_b"],
        )
        conversation = [
            {"role": "system", "content": SYS_PROMPT},
            {"role": "user", "content": user_message},
        ]
        text = tokenizer.apply_chat_template(
            conversation,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        prompts.append(text)
    return prompts


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config_path", type=str, required=True)
    args = parser.parse_args()

    cfg = OmegaConf.load(args.config_path)

    # Load predictions
    preds_a = pd.read_parquet(cfg.preds_a_path)
    preds_b = pd.read_parquet(cfg.preds_b_path)
    print(f"Preds A: {len(preds_a):,} | Preds B: {len(preds_b):,}")

    id_col = cfg.get("id_col", "id")
    df = preds_a[[id_col, "transliteration", "translation"]].merge(
        preds_b[[id_col, "translation"]],
        on=id_col,
        suffixes=("_a", "_b"),
    )
    print(f"Merged: {len(df):,} rows")

    # Load model
    print(f"Loading model: {cfg.model_path}")
    llm = vllm.LLM(
        model=cfg.model_path,
        max_model_len=cfg.get("max_model_len", 2048),
        max_num_seqs=cfg.get("max_num_seqs", 32),
        dtype=cfg.get("dtype", "half"),
        cpu_offload_gb=cfg.get("cpu_offload_gb", 0),
        swap_space=cfg.get("swap_space", 0),
        trust_remote_code=True,
        tensor_parallel_size=torch.cuda.device_count(),
        enforce_eager=True,
    )

    tokenizer = llm.get_tokenizer()
    prompts = build_prompts(df, tokenizer)
    print(f"Built {len(prompts):,} prompts")

    # Generate with logprobs
    outputs = llm.generate(
        prompts,
        vllm.SamplingParams(temperature=0.0, max_tokens=1, logprobs=20),
        use_tqdm=True,
    )

    # Extract probabilities
    probs_list = []
    for output in outputs:
        lps = output.outputs[0].logprobs[0]
        probs_list.append(get_probs(lps))

    probs_arr = np.array(probs_list)
    df["prob_A"] = probs_arr[:, 0]
    df["prob_B"] = probs_arr[:, 1]
    df["prob_EQUAL"] = probs_arr[:, 2]

    # Pick: switch to B only if prob_B >= threshold, otherwise keep A
    b_threshold = cfg.b_threshold
    df["pick"] = np.where(df["prob_B"] >= b_threshold, "B", "A")
    df["translation"] = np.where(df["pick"] == "B", df["translation_b"], df["translation_a"])
    print(f"\nb_threshold={b_threshold}")

    # Stats
    n = len(df)
    for label in ["A", "B"]:
        cnt = (df["pick"] == label).sum()
        print(f"  {label}: {cnt:,} ({cnt / n * 100:.1f}%)")

    print(f"\n{df[['pick', 'prob_A', 'prob_B', 'prob_EQUAL']].head(20).to_string()}")

    for col in ["prob_A", "prob_B", "prob_EQUAL"]:
        v = df[col]
        print(f"  {col}: mean={v.mean():.3f}  std={v.std():.3f}  min={v.min():.3f}  max={v.max():.3f}")

    # Save
    output_path = Path(cfg.output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(output_path, index=False)
    print(f"Saved {len(df):,} rows -> {output_path}")

    # Submission
    sub_df = df[[id_col, "translation"]].copy()
    sub_path = output_path.parent / "submission.csv"
    sub_df.to_csv(sub_path, index=False)
    print(f"Submission -> {sub_path}")


if __name__ == "__main__":
    main()

Writing run_rm.py


In [11]:
%%writefile conf_rm.yaml

model_path: /kaggle/input/models/conjuring92/dpc_qwen8b_rm/transformers/default/2
preds_a_path: ./model_outputs/pred_m1.parquet
preds_b_path: ./model_outputs/pred_m3.parquet
output_path: ./model_outputs/reward_picks.parquet
id_col: id

max_model_len: 2048
max_num_seqs: 32
cpu_offload_gb: 0
swap_space: 0
dtype: half

b_threshold: 0.65

Writing conf_rm.yaml


In [12]:
%%time
!python run_rm.py --config_path conf_rm.yaml

Preds A: 4 | Preds B: 4
Merged: 4 rows
Loading model: /kaggle/input/models/conjuring92/dpc_qwen8b_rm/transformers/default/2
2026-03-22 20:51:57.542889: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774212717.565131     164 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774212717.574639     164 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774212717.592466     164 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774212717.592495     164 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid lin

# Submission

In [13]:
import pandas as pd

test_df = pd.read_csv("/kaggle/input/deep-past-initiative-machine-translation/test.csv")
pred_df = pd.read_parquet("./model_outputs/reward_picks.parquet")

# df_m1 = pd.read_parquet("./model_outputs/pred_m1.parquet")
# df_m2 = pd.read_parquet("./model_outputs/pred_m2.parquet")

In [14]:
submission = pd.DataFrame({"id": test_df["id"].values.tolist(), "translation": pred_df['translation'].values.tolist()})
submission["translation"] = submission["translation"].apply(lambda x: x if len(x) > 0 else "broken text")
submission["translation"] = submission["translation"].apply(lambda x: x.replace("<unknown>", "<gap>"))

submission.to_csv("submission.csv", index=False)

In [15]:
pd.options.display.max_colwidth = None
submission.head()

,id,translation
0,0,"From the Kanesh colony to the dātum-payers, our messengers, every single colony and the trading stations: A tablet came to the City."
1,1,"According to the tablet of the City, until this day no one who bought my meteoric iron is entitled to part of the profit. The kārum of Kalia will collect my tithe."
2,2,"As soon as you have heard our tablet, there, whether he sold it to the palace, or he showed it to the orders of the palace, or he is carrying it without having yet sold it, as much meteoric iron as he is carrying, write the exact amount of whose name and the name of my father on a tablet and send it to me with our messenger."
3,3,"I sent the copy of our tablet to every single colony and to every single trading station. Whether he sold the meteoric iron for merchants, write the name of the man."


In [16]:
# df_m1

In [17]:
# df_m2